# Bayesian Credibility Inference

Owner: Member 5

This notebook combines extraction confidence, entity-linking confidence, KG reasoning status, KG confidence, and optional LIAR speaker-history metadata into final credibility probabilities.

## Role in the Project

Notebook 4 produces `04_KG_Reasoning/kg_results.json`, where each claim is marked as `supported`, `contradicted`, or `unknown` by conservative KG rules. This notebook converts that symbolic result into `P(true)`, `P(false)`, and a final verdict for the downstream report and responsible AI reflection.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'LIAR_dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / '05_Bayesian_Inference'
sys.path.insert(0, str(MODULE_DIR))

from run_bayesian_inference import run_bayesian_inference

INPUT_PATH = PROJECT_ROOT / '04_KG_Reasoning' / 'kg_results.json'
LINKED_INPUT_PATH = PROJECT_ROOT / '03_Entity_Linking_KR' / 'linked_entities.json'
OUTPUT_PATH = MODULE_DIR / 'final_verdicts.json'
SUMMARY_PATH = MODULE_DIR / 'final_verdict_summary.json'

INPUT_PATH, LINKED_INPUT_PATH, OUTPUT_PATH, SUMMARY_PATH

(PosixPath('/Users/shanusharma/AIaP-Group_Project/04_KG_Reasoning/kg_results.json'),
 PosixPath('/Users/shanusharma/AIaP-Group_Project/03_Entity_Linking_KR/linked_entities.json'),
 PosixPath('/Users/shanusharma/AIaP-Group_Project/05_Bayesian_Inference/final_verdicts.json'),
 PosixPath('/Users/shanusharma/AIaP-Group_Project/05_Bayesian_Inference/final_verdict_summary.json'))

## Input Preview

The preferred input is the KG reasoning output from Notebook 4. The Bayesian runner also reads Notebook 3's linked-entity output when available so that extraction and linking confidence can be used to scale evidence strength.

In [2]:
kg_records = json.loads(INPUT_PATH.read_text())
kg_records[:3]

[{'claim_id': 'claim-00001',
  'raw_claim': 'Says the Annies List political group supports third-trimester abortions on demand.',
  'kg_status': 'unknown',
  'kg_confidence': 0.25,
  'evidence': 'No subject or object KG entity was linked, so KG reasoning cannot verify the claim.',
  'reasoning_rule': 'insufficient_kg_evidence',
  'source': 'local KG reasoning rules',
  'label': 'false',
  'subject': 'Annies List',
  'subject_kg_id': None,
  'subject_kg_label': None,
  'relation': 'say',
  'property_id': None,
  'object': 'demand',
  'object_kg_id': None,
  'object_kg_label': None,
  'linking_confidence': 0.224},
 {'claim_id': 'claim-00002',
  'raw_claim': 'When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration.',
  'kg_status': 'unknown',
  'kg_confidence': 0.4,
  'evidence': 'Linked entities are available, but no safe KG rule applies to this extracted relation.',
  'reasoning_rule': 'insufficient_kg_

## Bayesian Method

The model is intentionally conservative:

- start from a neutral `P(true)=0.50` prior
- adjust the prior only slightly with LIAR speaker-history counts when available
- treat KG support as positive evidence and KG contradiction as negative evidence
- scale KG evidence by extraction confidence, entity-linking confidence, and KG confidence
- keep `unknown` KG results near the prior instead of forcing every claim into true or false

In [3]:
results = run_bayesian_inference(PROJECT_ROOT)
summary = results['summary']
summary

{'input_path': '04_KG_Reasoning/kg_results.json',
 'linked_input_path': '03_Entity_Linking_KR/linked_entities.json',
 'output_path': '05_Bayesian_Inference/final_verdicts.json',
 'report_table_path': 'report_assets/tables/bayesian_verdict_distribution.csv',
 'total_claims': 500,
 'verdict_counts': {'uncertain': 499, 'likely true': 1},
 'kg_status_counts': {'unknown': 499, 'supported': 1},
 'average_probability_true': 0.485,
 'average_decision_confidence': 0.039,
 'average_evidence_weight': 0.207,
 'model': {'prior': 'neutral 0.50 adjusted only slightly by LIAR speaker-history counts when available',
  'kg_log_bayes_factors': {'supported': 2.4,
   'contradicted': -2.4,
   'unknown': 0.0},
  'verdict_threshold': 0.65,
  'speaker_prior_strength': 0.25},
 'reference_bucket_alignment': {'available_claims': 500,
  'alignment_rate': 0.392,
  'note': 'Reference labels are used for evaluation only, not to compute final probabilities.',
  'confusion': [{'reference_bucket': 'likely false',
    'f

## Output Preview

Each output record contains the required final-verdict fields: `claim_id`, `probability_true`, `probability_false`, `final_verdict`, `decision_confidence`, and `reasoning`. Extra context is retained for traceability.

In [4]:
final_verdicts = results['final_verdicts']
final_verdicts[:5]

[{'claim_id': 'claim-00001',
  'raw_claim': 'Says the Annies List political group supports third-trimester abortions on demand.',
  'probability_true': 0.499,
  'probability_false': 0.501,
  'final_verdict': 'uncertain',
  'decision_confidence': 0.002,
  'reasoning': "Started with speaker-history prior from LIAR credit counts (positive=0.0, negative=1.0, total=1). KG status 'unknown' at confidence 0.25 adds no directional truth signal; posterior probability stays near 0.50, so no binary verdict is forced.",
  'probability_prior_true': 0.499,
  'evidence_weight': 0.157,
  'kg_status': 'unknown',
  'kg_confidence': 0.25,
  'kg_evidence': 'No subject or object KG entity was linked, so KG reasoning cannot verify the claim.',
  'reasoning_rule': 'insufficient_kg_evidence',
  'extraction_confidence': 0.895,
  'linking_confidence': 0.224,
  'speaker': 'dwayne-bohac',
  'reference_label': 'false'},
 {'claim_id': 'claim-00002',
  'raw_claim': 'When did the decline of coal start? It started when

## Verdict Distribution

This distribution supports the empirical analysis section of the report. A large `uncertain` count is expected when KG reasoning does not have enough evidence to support or contradict most claims.

In [5]:
from collections import Counter

Counter(record['final_verdict'] for record in final_verdicts)

Counter({'uncertain': 499, 'likely true': 1})

## High-Confidence Examples

The rows below are the strongest Bayesian decisions. They should be inspected in the report because high confidence depends on the upstream extraction, linking, and KG evidence quality.

In [6]:
preview_fields = [
    'claim_id', 'final_verdict', 'probability_true', 'probability_false',
    'decision_confidence', 'kg_status', 'kg_confidence', 'reference_label', 'raw_claim'
]

[
    {field: record.get(field) for field in preview_fields}
    for record in sorted(final_verdicts, key=lambda item: item['decision_confidence'], reverse=True)[:10]
]

[{'claim_id': 'claim-00037',
  'final_verdict': 'likely true',
  'probability_true': 0.77,
  'probability_false': 0.23,
  'decision_confidence': 0.541,
  'kg_status': 'supported',
  'kg_confidence': 0.671,
  'reference_label': 'true',
  'raw_claim': 'Austin is a city that has basically doubled in size every 25 years or so since it was founded.'},
 {'claim_id': 'claim-00047',
  'final_verdict': 'uncertain',
  'probability_true': 0.391,
  'probability_false': 0.609,
  'decision_confidence': 0.218,
  'kg_status': 'unknown',
  'kg_confidence': 0.4,
  'reference_label': 'pants-fire',
  'raw_claim': 'Obamacare will provide insurance to all non-U.S. residents, even if they are here illegally.'},
 {'claim_id': 'claim-00163',
  'final_verdict': 'uncertain',
  'probability_true': 0.391,
  'probability_false': 0.609,
  'decision_confidence': 0.218,
  'kg_status': 'unknown',
  'kg_confidence': 0.3,
  'reference_label': 'half-true',
  'raw_claim': 'While Sarah was Mayor of Wasilla she tried to fire

## Reference-Label Check

`reference_label` comes from the LIAR dataset and is used only for evaluation. It is not used to compute `P(true)` or the final verdict.

In [7]:
summary.get('reference_bucket_alignment')

{'available_claims': 500,
 'alignment_rate': 0.392,
 'note': 'Reference labels are used for evaluation only, not to compute final probabilities.',
 'confusion': [{'reference_bucket': 'likely false',
   'final_verdict': 'uncertain',
   'count': 129},
  {'reference_bucket': 'likely true',
   'final_verdict': 'likely true',
   'count': 1},
  {'reference_bucket': 'likely true',
   'final_verdict': 'uncertain',
   'count': 175},
  {'reference_bucket': 'uncertain',
   'final_verdict': 'uncertain',
   'count': 195}]}

## Report Handoff

The runner writes `final_verdicts.json`, `final_verdict_summary.json`, and `report_assets/tables/bayesian_verdict_distribution.csv`. These files provide the final credibility results for the empirical analysis and conclusion sections.

In [8]:
json.loads(SUMMARY_PATH.read_text())

{'input_path': '04_KG_Reasoning/kg_results.json',
 'linked_input_path': '03_Entity_Linking_KR/linked_entities.json',
 'output_path': '05_Bayesian_Inference/final_verdicts.json',
 'report_table_path': 'report_assets/tables/bayesian_verdict_distribution.csv',
 'total_claims': 500,
 'verdict_counts': {'uncertain': 499, 'likely true': 1},
 'kg_status_counts': {'unknown': 499, 'supported': 1},
 'average_probability_true': 0.485,
 'average_decision_confidence': 0.039,
 'average_evidence_weight': 0.207,
 'model': {'prior': 'neutral 0.50 adjusted only slightly by LIAR speaker-history counts when available',
  'kg_log_bayes_factors': {'supported': 2.4,
   'contradicted': -2.4,
   'unknown': 0.0},
  'verdict_threshold': 0.65,
  'speaker_prior_strength': 0.25},
 'reference_bucket_alignment': {'available_claims': 500,
  'alignment_rate': 0.392,
  'note': 'Reference labels are used for evaluation only, not to compute final probabilities.',
  'confusion': [{'reference_bucket': 'likely false',
    'f

## Limitations

The Bayesian output is only as reliable as the upstream evidence. Most KG results are `unknown` because the project uses conservative local reasoning rules, so many claims remain uncertain. Speaker-history priors are deliberately weak because they can encode political and historical bias. The model therefore favours transparent uncertainty over overconfident misinformation labels.